# AI CCTV Surveillance — Phase 1 POC
## Kandhamal District Police | Starlight Data Solutions

## Step 1: Setup

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

# Clone repo and set working directory
import os
if not os.path.exists('odisha-dist-cctv'):
    !git clone https://github.com/saiesh2401/odisha-dist-cctv.git
os.chdir('odisha-dist-cctv/poc')

# Download plate detector model
if not os.path.exists('models/license_plate_detector.pt'):
    !git clone --depth 1 https://github.com/sveyek/Video-ANPR.git /tmp/video-anpr 2>/dev/null
    !mkdir -p models && cp /tmp/video-anpr/models/license_plate_detector.pt models/
    !rm -rf /tmp/video-anpr

# Create directories
!mkdir -p results uploads wanted_persons

print(f"\nWorking dir: {os.getcwd()}")
required = ['engine.py', 'server.py', 'stolen_vehicles_db.json', 'models/license_plate_detector.pt']
for f in required:
    status = '✅' if os.path.exists(f) else '❌ MISSING'
    print(f"  {f}: {status}")

In [ ]:
!pip install -q ultralytics easyocr face_recognition fastapi uvicorn python-multipart pillow opencv-python-headless

## Step 2: Initialize Engine

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import warnings
warnings.filterwarnings('ignore')

from engine import SurveillanceEngine
engine = SurveillanceEngine()
engine.anpr._ensure_models()

## Step 3: UC-1 & UC-2 — Database Matching

In [ ]:
plates = ["OD02AK7834", "OD21F9012", "MH12AB1234", "OD21C7890", "WB02X5678", "OD21A1111", "OD05M8901", "OD21D9999"]

print(f"{'Plate':<18} | Result")
print('-' * 65)
for plate in plates:
    r = engine.anpr.match_plate(plate)
    if r['matches']:
        for m in r['matches']:
            print(f"{plate:<18} | {m['severity']:<8} → {m['type']}")
    else:
        print(f"{plate:<18} | CLEAR")

## Step 4: ANPR on Demo Images

In [ ]:
from IPython.display import Image, display
import glob

# Process all demo images
demo_images = sorted(glob.glob('data/demo_*.jpg'))
print(f"Found {len(demo_images)} demo images\n")

for img_path in demo_images:
    print(f"\n{'='*50}")
    print(f"Processing: {img_path}")
    result = engine.process_anpr(img_path)
    print(f"Vehicles: {result.get('vehicles_detected', 0)} | Plates: {result.get('plates_read', 0)} | Alerts: {len(result.get('alerts', []))} | Time: {result.get('processing_time_sec')}s")
    
    for p in result.get('plate_details', []):
        print(f"  Plate: {p['plate_text']}")
    for a in result.get('alerts', []):
        for m in a['matches']:
            print(f"  🚨 [{m['severity']}] {m['type']}: {a['plate']}")
    
    if result.get('annotated_image'):
        display(Image(filename=result['annotated_image'], width=600))

## Step 5: ANPR on CCTV Video

In [ ]:
from IPython.display import Image, display
import glob, time

# Find video files
videos = glob.glob('data/*.mp4') + glob.glob('data/*.avi')
print(f"Found {len(videos)} videos: {videos}\n")

if videos:
    video_path = videos[0]
    print(f"Processing: {video_path}")
    t0 = time.time()
    result = engine.anpr.process_video(video_path, sample_every_n_frames=15)
    
    print(f"\nFrames: {result['frames_processed']}/{result['total_frames']}")
    print(f"Time: {time.time()-t0:.1f}s | FPS: {result['fps_processed']}")
    print(f"Plates: {len(result['unique_plates_detected'])} | Alerts: {len(result['alerts'])}")
    
    print(f"\nPlates found:")
    for p in result['unique_plates_detected']:
        print(f"  {p}")
    
    print(f"\nAlerts:")
    for a in result['alerts']:
        for m in a['matches']:
            print(f"  🚨 [{m['severity']}] {m['type']}: {a['plate']} @ {a['timestamp_sec']}s")
        if a.get('evidence_image'):
            display(Image(filename=a['evidence_image'], width=600))
else:
    print("No video files found in data/. Place a .mp4 file there.")

## Step 6: UC-3 — Face Recognition

### Enroll wanted persons from Kaggle FRS dataset

In [ ]:
import shutil
from pathlib import Path

# Auto-enroll from FRS dataset if available
frs_base = Path('data/frs_samples')
if frs_base.exists():
    sample_dir = [d for d in frs_base.iterdir() if d.is_dir()][0]
    names = {'user_1':'Rajesh_Kumar','user_2':'Sanjay_Pradhan','user_3':'Bikram_Nayak',
             'user_4':'Dilip_Sahu','user_6':'Mohan_Behera','user_9':'Ashok_Panda',
             'user_10':'Suresh_Jena','user_11':'Rakesh_Mohanty','user_15':'Vikram_Das',
             'user_17':'Pradeep_Sahoo','user_18':'Anil_Swain'}
    
    for user_dir in sorted(sample_dir.iterdir()):
        if user_dir.is_dir() and user_dir.name in names:
            selfies = sorted((user_dir / 'selfies').glob('*'))
            if selfies:
                dest = f"wanted_persons/{names[user_dir.name]}.jpeg"
                shutil.copy2(str(selfies[0]), dest)
    print("Enrolled from FRS dataset")
else:
    print("No FRS dataset found. Enroll manually using next cell.")

count = engine.face.load_wanted_persons()
print(f"\nTotal wanted persons: {count}")

### Test face matching against archive photos

In [ ]:
from pathlib import Path

frs_base = Path('data/frs_samples')
if not frs_base.exists():
    print("No FRS dataset. Skip this cell.")
else:
    sample_dir = [d for d in frs_base.iterdir() if d.is_dir()][0]
    names = {'user_1':'Rajesh Kumar','user_2':'Sanjay Pradhan','user_3':'Bikram Nayak',
             'user_4':'Dilip Sahu','user_6':'Mohan Behera','user_9':'Ashok Panda',
             'user_10':'Suresh Jena','user_11':'Rakesh Mohanty','user_15':'Vikram Das',
             'user_17':'Pradeep Sahoo','user_18':'Anil Swain'}

    correct, total = 0, 0
    print(f"{'User':<10} {'Expected':<20} {'Got':<20} {'Conf':>6} {'Time':>6} Status")
    print('-' * 80)
    
    for user_dir in sorted(sample_dir.iterdir()):
        if not user_dir.is_dir() or user_dir.name not in names:
            continue
        expected = names[user_dir.name]
        if expected not in engine.face.wanted_persons:
            continue
        archive = sorted((user_dir / 'archive_selfies').glob('*'))
        if not archive:
            continue
        
        import time
        t0 = time.time()
        result = engine.face.search_face(str(archive[0]))
        elapsed = time.time() - t0
        total += 1
        
        if result.get('matches'):
            m = result['matches'][0]
            ok = m['matched_person'] == expected
            if ok: correct += 1
            status = '✅' if ok else '❌'
            print(f"{user_dir.name:<10} {expected:<20} {m['matched_person']:<20} {m['confidence']:>5.1f}% {elapsed:>5.1f}s {status}")
        else:
            print(f"{user_dir.name:<10} {expected:<20} {'NO MATCH':<20} {'':>6} {elapsed:>5.1f}s ❌")
    
    print(f"\n{'='*40}")
    print(f"ACCURACY: {correct}/{total} = {correct/total*100:.1f}%" if total else "No tests run")

## Step 7: Run Full Acceptance Test (25 criteria)

In [ ]:
!python generate_acceptance_report.py

---
## Step 8: Launch Web App (with public URL)

This starts your full demo web app on Colab's GPU and gives you a public URL.
Open it on your phone, laptop, or share with anyone.

In [ ]:
!pip install -q pyngrok nest-asyncio

import nest_asyncio
nest_asyncio.apply()

from pyngrok import ngrok
import uvicorn

# Get free ngrok token at https://dashboard.ngrok.com/signup
# Uncomment and paste your token:
# ngrok.set_auth_token("YOUR_TOKEN_HERE")

public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"  YOUR WEB APP IS LIVE!")
print(f"  Public URL: {public_url}")
print(f"  Open this on ANY device — phone, laptop, share with SP")
print(f"{'='*60}\n")

from server import app
uvicorn.run(app, host="0.0.0.0", port=8000)